# RAG Stress-Test — Retriever Ablation (Colab)

Runs `faiss_only`, `bm25_only`, and `no_rerank` ablations across all 8 perturbation conditions on a GPU.

**Before running this notebook:**
1. Push your latest code to GitHub: `git push origin main`
2. Upload `corpus.db`, `faiss.index`, `questions.json` to a folder named `rag_stress_test_data` in your Google Drive root.
3. **Runtime → Change runtime type → GPU (T4 or better)**
4. Then **Runtime → Run all** and wait. Total time ~1-2 hr on T4.

# RAG Stress-Test — Retriever Ablation (Colab, FIXED)

Runs `faiss_only` and `no_rerank` ablations across all 8 conditions on a GPU.

**Fix vs prior run:** the MedCPT query encoder is forced onto CPU (FP32) so its
embeddings match the Mac-built FAISS index. CUDA kernels produced subtly
different floats that silently missed against the index, giving MAP=0 results.
Ollama (LLaMA) still uses GPU — only the small encoder runs on CPU (~2 sec each).

**Before running:**
1. On your Mac: push the latest code, `git push origin main`
2. Make sure `corpus.db`, `faiss.index`, `questions.json` already live in
   `MyDrive/rag_stress_test_data/` (from prior run, no need to re-upload)
3. Runtime → Change runtime type → T4 GPU → Save
4. Runtime → Run all

## 1. Mount Google Drive and verify data files exist

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA = '/content/drive/MyDrive/rag_stress_test_data'
for fname in ['corpus.db', 'faiss.index', 'questions.json']:
    p = os.path.join(DRIVE_DATA, fname)
    assert os.path.exists(p), f'MISSING: {p}'
    print(f'OK  {fname}: {os.path.getsize(p)/1e6:.1f} MB')

Mounted at /content/drive
OK  corpus.db: 892.2 MB
OK  faiss.index: 499.3 MB
OK  questions.json: 33.3 MB


## 2. Clone the repo from GitHub

In [2]:

%cd /content
!rm -rf rag_stress_test
!git clone https://github.com/Tanushree28/rag_stress_test.git
%cd rag_stress_test

/content
Cloning into 'rag_stress_test'...
remote: Enumerating objects: 889, done.
remote: Counting objects: 100% (889/889), done.
remote: Compressing objects: 100% (630/630), done.
remote: Total 889 (delta 221), reused 862 (delta 194), pack-reused 0 (from 0)
Receiving objects: 100% (889/889), 14.60 MiB | 16.48 MiB/s, done.
Resolving deltas: 100% (221/221), done.
/content/rag_stress_test


## 3. Copy data files from Drive into the repo

In [3]:
!mkdir -p data
!cp "/content/drive/MyDrive/rag_stress_test_data/corpus.db" data/
!cp "/content/drive/MyDrive/rag_stress_test_data/faiss.index" data/
!cp "/content/drive/MyDrive/rag_stress_test_data/questions.json" data/
!ls -lh data/

total 1.4G
-rw-r--r-- 1 root root 8.8M Apr 26 23:55 chunks.json
-rw------- 1 root root 851M Apr 26 23:55 corpus.db
-rw-r--r-- 1 root root  32K Apr 26 23:55 corpus.db-shm
-rw-r--r-- 1 root root 4.0M Apr 26 23:55 corpus.db-wal
-rw------- 1 root root 477M Apr 26 23:56 faiss.index
-rw-r--r-- 1 root root  32M Apr 26 23:56 questions.json


## 4. Install Python dependencies (~3 min)

In [4]:
!pip install -q faiss-cpu transformers sentence-transformers ollama scipy rouge-score numpy

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 56.2 MB/s eta 0:00:00


## 5. Force the encoder onto CPU (the fix)

This single env var causes `retriever.py` to load MedCPT and the cross-encoder
on CPU even though we have a GPU. Ollama (LLaMA generation) still uses GPU.

In [5]:
import os
os.environ['RAG_FORCE_CPU_ENCODER'] = '1'
print('RAG_FORCE_CPU_ENCODER =', os.environ['RAG_FORCE_CPU_ENCODER'])

RAG_FORCE_CPU_ENCODER = 1


## 5. Install + start Ollama, pull LLaMA 3.1:8b (~6 min one-time)

In [6]:
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 107 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (363 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-

In [7]:
import subprocess, time
# Start the ollama server in the background
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)
!ollama --version

ollama version is 0.21.2


In [8]:
!ollama pull llama3.1:8b

## 6. Smoke test — make sure the pipeline runs at least once

In [9]:
import sys; sys.path.insert(0, '.')
from retriever import retrieve
from generator import generate
passages = retrieve('What causes Hirschsprung disease?', top_k=3, mode='faiss_only')
print(f'Got {len(passages)} passages')
result = generate('What causes Hirschsprung disease?', passages)
print('Answer:', result['answer'][:200])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Got 3 passages
Answer: According to passage [1], recent advances show that the RET gene is a major locus involved in the pathogenesis of Hirschsprung disease. The study found that single nucleotide polymorphisms in exons 2,


## 7. SANITY CHECK — verify FAISS retrieval is working

If the rerank scores below are roughly the same as on Mac
(`hybrid` ~6.5, `faiss_only` ~2.0+, `no_rerank` ~0.6+), the fix worked.
If any are near zero, **STOP** — don't waste 5 hr running broken retrieval.

In [10]:
import sys; sys.path.insert(0, '.')
from retriever import retrieve

Q = 'What causes Hirschsprung disease?'
for mode in ['hybrid', 'faiss_only', 'bm25_only', 'no_rerank']:
    p = retrieve(Q, top_k=5, mode=mode)
    score = p[0].get('rerank_score') if p else None
    status = 'OK' if (score is not None and score > 0.5) else 'BROKEN'
    print(f'{status:<7} {mode:<12} top rerank={score}')

# Hard assertion -- abort the notebook if any mode is broken
for mode in ['hybrid', 'faiss_only', 'no_rerank']:
    p = retrieve(Q, top_k=5, mode=mode)
    assert p and p[0]['rerank_score'] > 0.4, f'BROKEN: {mode} top score is {p[0]["rerank_score"] if p else None}'
print('\nSANITY CHECK PASSED -- safe to proceed')

OK      hybrid       top rerank=6.483347415924072
OK      faiss_only   top rerank=2.250239849090576
OK      bm25_only    top rerank=6.483347415924072
OK      no_rerank    top rerank=0.6576457023620605

SANITY CHECK PASSED -- safe to proceed


## 8. Run the ablation (~1-2 hrs on T4)

Watch the progress lines: `[mode] q X/Y -- done=N -- ETA M min`.

In [11]:
!RAG_FORCE_CPU_ENCODER=1 python run_ablation.py --n_per_type 25 --modes faiss_only,no_rerank

Streaming output truncated to the last 5000 lines.
2026-04-27 00:21:41,730 INFO: Negation cache hit
2026-04-27 00:21:46,337 INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-27 00:21:46,337 INFO: Conflict injected: 3/5 passages negated (ratio=70%)
2026-04-27 00:21:47,083 INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-27 00:21:47,083 INFO: Generated answer (61 chars, 0 citations) for: Was modafinil tested for schizophrenia treatment?
Batches: 100% 1/1 [00:00<00:00, 36.29it/s]
2026-04-27 00:21:47,155 INFO: Unanswerable (partial): removed 0/0 answer-bearing passages, 5 remain
2026-04-27 00:21:47,957 INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-27 00:21:47,957 INFO: Generated answer (124 chars, 0 citations) for: Was modafinil tested for schizophrenia treatment?
Batches: 100% 1/1 [00:00<00:00, 29.00it/s]
2026-04-27 00:21:48,192 INFO: Unanswerable (full): removed 0/0 answer-bearing pass

In [11]:
# !python run_ablation.py --n_per_type 25 --modes faiss_only,bm25_only,no_rerank

Streaming output truncated to the last 5000 lines.
Batches: 100% 1/1 [00:00<00:00, 10.53it/s]
2026-04-26 19:50:03,166 INFO: Negation cache hit
2026-04-26 19:50:03,166 INFO: Negation cache hit
2026-04-26 19:50:14,544 INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-26 19:50:14,544 INFO: Conflict injected: 3/5 passages negated (ratio=70%)
2026-04-26 19:50:18,829 INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-26 19:50:18,830 INFO: Generated answer (212 chars, 2 citations) for: Unlike DNA, RNA is not methylated, yes or no?
Batches: 100% 1/1 [00:00<00:00, 18.28it/s]
2026-04-26 19:50:19,213 INFO: Unanswerable (partial): removed 2/4 answer-bearing passages, 3 remain
2026-04-26 19:50:20,926 INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-26 19:50:20,926 INFO: Generated answer (7 chars, 1 citations) for: Unlike DNA, RNA is not methylated, yes or no?
2026-04-26 19:50:21,098 INFO: Unanswerable

In [ ]:
# !python run_ablation.py --n_per_type 25 --modes faiss_only,bm25_only,no_rerank

Streaming output truncated to the last 5000 lines.
2026-04-25 23:34:57,643 INFO: Conflict injected: 3/5 passages negated (ratio=70%)
2026-04-25 23:35:02,083 INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-25 23:35:02,084 INFO: Generated answer (231 chars, 2 citations) for: Unlike DNA, RNA is not methylated, yes or no?
Batches: 100% 1/1 [00:00<00:00, 27.12it/s]
2026-04-25 23:35:02,453 INFO: Unanswerable (partial): removed 2/4 answer-bearing passages, 3 remain
2026-04-25 23:35:03,940 INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-25 23:35:03,940 INFO: Generated answer (7 chars, 1 citations) for: Unlike DNA, RNA is not methylated, yes or no?
2026-04-25 23:35:04,102 INFO: Unanswerable (full): removed 4/4 answer-bearing passages, 1 remain
2026-04-25 23:35:06,464 INFO: HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-04-25 23:35:06,466 INFO: Generated answer (213 chars, 0 citations) for: Unlike DN

## 8. Save results back to Drive AND offer direct download

In [12]:
import shutil, os
src = 'data/experiments_ablation.db'
assert os.path.exists(src), 'No results -- did the run finish?'
shutil.copy(src, '/content/drive/MyDrive/rag_stress_test_data/experiments_ablation_v4.db')
print(f'Copied to Drive: rag_stress_test_data/experiments_ablation_v4.db')
print(f'Size: {os.path.getsize(src)/1e6:.2f} MB')
from google.colab import files
files.download(src)

Copied to Drive: rag_stress_test_data/experiments_ablation_v3.db
Size: 14.23 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## On your Mac, after download:

Move the downloaded `experiments_ablation.db` into your repo's `data/` folder, then merge:

```bash
sqlite3 data/experiments.db \
  "ATTACH 'data/experiments_ablation.db' AS abl; \
   INSERT INTO experiments \
     (timestamp, question_id, question_type, question_body, condition, \
      answer, metrics, passages, duration_s, retriever_mode) \
   SELECT timestamp, question_id, question_type, question_body, condition, \
          answer, metrics, passages, duration_s, retriever_mode \
   FROM abl.experiments;"
```

Then refresh `http://localhost:3000/analytics` and use the **Retriever** dropdown to compare modes.